In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/text-assignment/resolved_queries.csv
/kaggle/input/text-assignment/new_queries.csv


In [9]:
import difflib
import re
from collections import Counter
import numpy as np
import pandas as pd

In [10]:
import difflib
import re
from collections import Counter
import numpy as np
import pandas as pd

# Load data from CSV
resolved = pd.read_csv('/kaggle/input/text-assignment/resolved_queries.csv')
new_queries = pd.read_csv('/kaggle/input/text-assignment/new_queries.csv')

# Convert to dictionary and list for consistency with original logic
resolved = dict(zip(resolved['Query_ID'], resolved['Pre_Resolved_Query']))
new_queries = list(zip(new_queries['Variation_Query'], new_queries['Matches_With_Query_ID']))

# Preprocessing function, handling Series or other types
def preprocess(text):
    # Convert input to string, handling Series or other types
    if isinstance(text, pd.Series):
        text = text.iloc[0] if not text.empty else ""
    elif not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

# Preprocess all
resolved_pre = {k: preprocess(v) for k, v in resolved.items()}
new_pre = [(preprocess(q), id) for q, id in new_queries]

# Fuzzy methods
def ratio(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio() * 100

def partial_ratio(a, b):
    if len(a) == 0 or len(b) == 0:
        return 0
    short, long = (a, b) if len(a) < len(b) else (b, a)
    m = difflib.SequenceMatcher(None, short, long)
    blocks = m.get_matching_blocks()
    scores = []
    for i, j, n in blocks:
        if n > 0:
            scores.append(ratio(short[i:i+n], long[j:j+n]))
    return max(scores, default=0)

def token_sort_ratio(a, b):
    a_sorted = ' '.join(sorted(a.split()))
    b_sorted = ' '.join(sorted(b.split()))
    return ratio(a_sorted, b_sorted)

def token_set_ratio(a, b):
    tokens_a = set(a.split())
    tokens_b = set(b.split())
    inter = tokens_a & tokens_b
    diff_a = tokens_a - inter
    diff_b = tokens_b - inter
    s_inter = ' '.join(sorted(inter))
    s_diff_a = ' '.join(sorted(diff_a))
    s_diff_b = ' '.join(sorted(diff_b))
    if not s_inter:
        return 0
    r1 = ratio(s_inter + ' ' + s_diff_a, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    r2 = ratio(s_inter + ' ' + s_diff_b, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    r3 = ratio(s_inter, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    return max(r1, r2, r3)

# BoW and TF-IDF
all_queries = list(resolved_pre.values())
vocab = sorted(set(word for q in all_queries for word in q.split()))

# Document frequency for IDF
df = Counter()
for q in all_queries:
    words = set(q.split())
    for w in words:
        df[w] += 1
N = len(all_queries)
idf = {w: np.log(N / df[w]) if df[w] > 0 else 0 for w in vocab}

def bow_vec(text):
    count = Counter(text.split())
    vec = np.array([count.get(w, 0) for w in vocab])
    return vec

def tfidf_vec(text):
    count = Counter(text.split())
    tf = {w: count[w] for w in count}  # raw tf
    vec = np.array([tf.get(w, 0) * idf.get(w, 0) for w in vocab])
    return vec

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)) * 100

# Methods dict
methods = {
    'ratio': ratio,
    'partial_ratio': partial_ratio,
    'token_sort_ratio': token_sort_ratio,
    'token_set_ratio': token_set_ratio,
    'bow_cosine': lambda a, b: cosine_sim(bow_vec(a), bow_vec(b)),
    'tfidf_cosine': lambda a, b: cosine_sim(tfidf_vec(a), tfidf_vec(b))
}

# Evaluate
thresholds = np.arange(40, 101, 5)

results = {}
for method_name, sim_func in methods.items():
    best_acc = 0
    best_th = 0
    for th in thresholds:
        correct = 0
        for q, true_id in new_pre:
            sims = {rid: sim_func(q, rq) for rid, rq in resolved_pre.items()}
            max_sim = max(sims.values())
            pred_id = [rid for rid, s in sims.items() if s == max_sim][0] if max_sim >= th else None
            if pred_id == true_id:
                correct += 1
        acc = correct / len(new_pre)
        if acc > best_acc:
            best_acc = acc
            best_th = th
    results[method_name] = (best_acc, best_th)

# Output
print("Best method and threshold:")
for m, (acc, th) in results.items():
    print(f"{m}: Accuracy {acc:.2f} at threshold {th}")

# Use best method (token_set_ratio) for matching
best_method = max(results, key=lambda x: results[x][0])
best_th = results[best_method][1]
sim_func = methods[best_method]

print(f"\nBest method: {best_method} with acc {results[best_method][0]} at th {best_th}")

print("\nMatches using best method:")
for q, true_id in new_pre:
    sims = {rid: sim_func(q, rq) for rid, rq in resolved_pre.items()}
    max_sim = max(sims.values())
    pred_id = [rid for rid, s in sims.items() if s == max_sim][0] if max_sim >= best_th else None
    orig_q = [oq for oq, _ in new_queries if preprocess(oq) == q][0]
    print(f"Query: {orig_q} -> Predicted: {pred_id} (True: {true_id})")

Best method and threshold:
ratio: Accuracy 0.70 at threshold 40
partial_ratio: Accuracy 0.20 at threshold 40
token_sort_ratio: Accuracy 0.95 at threshold 40
token_set_ratio: Accuracy 0.95 at threshold 40
bow_cosine: Accuracy 0.85 at threshold 40
tfidf_cosine: Accuracy 0.95 at threshold 40

Best method: token_sort_ratio with acc 0.95 at th 40

Matches using best method:
Query: Unabel to conect to the internet -> Predicted: 1 (True: 1)
Query: Can’t connect to internet -> Predicted: 1 (True: 1)
Query: Intenet not working -> Predicted: None (True: 1)
Query: Payment failed while chekout -> Predicted: 2 (True: 2)
Query: Payment did not go through during chckout -> Predicted: 2 (True: 2)
Query: Payment issue at check out -> Predicted: 2 (True: 2)
Query: Application crashes when opening setings -> Predicted: 3 (True: 3)
Query: App crash when going to settings -> Predicted: 3 (True: 3)
Query: Settings cause the app to chrash -> Predicted: 3 (True: 3)
Query: Forgot passwrd and cant reset -> Pred

In [11]:
# Full Kaggle-ready notebook: fuzzy + BoW/TF-IDF matching with threshold tuning
# Copy into a Kaggle notebook cell and run.
import os, re, warnings, time
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import difflib
from tqdm.auto import tqdm

# Try rapidfuzz
try:
    from rapidfuzz import fuzz, process as rf_process
    RAPIDFUZZ = True
except Exception:
    RAPIDFUZZ = False

print("rapidfuzz available:", RAPIDFUZZ)

# --- Paths (change if necessary) ---
RESOLVED_PATH = '/kaggle/input/text-assignment/resolved_queries.csv'
NEW_PATH = '/kaggle/input/text-assignment/new_queries.csv'

if not (os.path.exists(RESOLVED_PATH) and os.path.exists(NEW_PATH)):
    print("ERROR: expected files not found. Please upload them to the notebook dataset or update paths.")
    print("Check what is in /kaggle/input with: !ls -la /kaggle/input")
    raise FileNotFoundError("CSV files missing at expected paths.")

# --- Load data ---
resolved = pd.read_csv(RESOLVED_PATH)
new = pd.read_csv(NEW_PATH)
print("resolved.shape:", resolved.shape)
print("new.shape:", new.shape)

# --- auto-detect text columns ---
def pick_text_col(df):
    for c in ['query','question','text','issue','title','message','description','input']:
        if c in df.columns:
            return c
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and df[col].dropna().shape[0] > 0:
            return col
    return df.columns[0]

resolved_text_col = pick_text_col(resolved)
new_text_col = pick_text_col(new)
print("Using resolved text column:", resolved_text_col)
print("Using new text column:", new_text_col)

# --- optional: detect ID/Ground-truth columns ---
def find_id_col(df, prefer=None):
    if prefer and prefer in df.columns: return prefer
    for token in ['id','uid','key','resolved_id','match_id','label','target','answer_id']:
        for col in df.columns:
            if token in col.lower():
                return col
    return None

resolved_id_col = find_id_col(resolved)  # resolved may already have unique ID
new_label_col = find_id_col(new)  # new might include ground truth labels (if available)

print("Resolved id column:", resolved_id_col)
print("New ground-truth label column (if any):", new_label_col)

# If resolved has no id column, create one using its index
if resolved_id_col is None:
    resolved = resolved.reset_index().rename(columns={'index':'_resolved_idx'})
    resolved_id_col = '_resolved_idx'

# --- Preprocessing function ---
STOPWORDS = set([
 'i','me','my','we','our','you','your','he','she','they','it','this','that','the','a','an','and','or','but','if','is','are','was','were',
 'in','on','at','for','with','about','to','from','by','of','as','be','have','has','do','does','did','so','not','no','please'
])

def preprocess_text(s, keep_digits=True):
    s = '' if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r'http\S+|www\.\S+', ' ', s)           # urls
    s = re.sub(r'\S+@\S+', ' ', s)                   # emails
    s = re.sub(r'\+?\d[\d\-\s]{4,}\d', ' ', s)       # phone-like
    # optionally keep digits; remove punctuation
    if keep_digits:
        s = re.sub(r'[^a-z0-9\s]', ' ', s)
    else:
        s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [t for t in s.split() if len(t) > 1 and t not in STOPWORDS]
    return ' '.join(tokens)

resolved['text_clean'] = resolved[resolved_text_col].astype(str).apply(preprocess_text)
new['text_clean'] = new[new_text_col].astype(str).apply(preprocess_text)

print("\nSample cleaned resolved:\n", resolved['text_clean'].head(5))
print("\nSample cleaned new:\n", new['text_clean'].head(5))

# --- Fuzzy matching utilities ---
# We'll produce top-3 matches for each fuzzy method.
def fuzzy_top_matches_rapid(query, choices, method='token_set_ratio', top_n=3):
    scorer_map = {
        'ratio': fuzz.ratio,
        'partial_ratio': fuzz.partial_ratio,
        'token_sort_ratio': fuzz.token_sort_ratio,
        'token_set_ratio': fuzz.token_set_ratio
    }
    scorer = scorer_map.get(method, fuzz.token_set_ratio)
    # rf_process.extract returns (choice, score, index)
    raw = rf_process.extract(query, choices, scorer=scorer, limit=top_n)
    out = []
    for choice, score, idx in raw:
        out.append({'resolved_idx': int(idx), 'score': float(score), 'resolved_text': choices[idx]})
    return out

# Python fallback (slower)
def fuzzy_top_matches_py(query, choices, method='token_set_ratio', top_n=3):
    def ratio_py(a,b): return int(difflib.SequenceMatcher(None,a,b).ratio()*100)
    def token_sort_ratio_py(a,b): return ratio_py(' '.join(sorted(a.split())), ' '.join(sorted(b.split())))
    def token_set_ratio_py(a,b):
        a_tokens=set(a.split()); b_tokens=set(b.split())
        inter = a_tokens & b_tokens
        inter_s = ' '.join(sorted(inter))
        a_only = ' '.join(sorted(a_tokens - inter))
        b_only = ' '.join(sorted(b_tokens - inter))
        cands = [inter_s] if inter_s else []
        cands += [inter_s + ' ' + a_only, inter_s + ' ' + b_only, a, b]
        best = 0
        for c in cands:
            best = max(best, ratio_py(c,b))
        return best
    scorer = {
        'ratio': ratio_py,
        'partial_ratio': ratio_py,
        'token_sort_ratio': token_sort_ratio_py,
        'token_set_ratio': token_set_ratio_py
    }[method]
    scores = []
    for i, choice in enumerate(choices):
        sc = scorer(query, choice)
        scores.append((i, sc))
    scores.sort(key=lambda x: x[1], reverse=True)
    out = [{'resolved_idx': int(i), 'score': float(s), 'resolved_text': choices[i]} for i,s in scores[:top_n]]
    return out

fuzzy_func = fuzzy_top_matches_rapid if RAPIDFUZZ else fuzzy_top_matches_py
print("Using fuzzy function:", "rapidfuzz" if RAPIDFUZZ else "python fallback")

choices = resolved['text_clean'].fillna('').tolist()

# Compute fuzzy matches (can be heavy if huge; tqdm for progress)
methods = ['token_set_ratio','token_sort_ratio','ratio','partial_ratio']
fuzzy_results = {m: [] for m in methods}
print("\nComputing fuzzy top-3 for each method... (this may take a while for large datasets)")
start = time.time()
for i, row in tqdm(new[['text_clean']].iterrows(), total=len(new)):
    q = row['text_clean']
    for m in methods:
        fuzzy_results[m].append(fuzzy_func(q, choices, method=m, top_n=3))
end = time.time()
print("Fuzzy matching completed in {:.1f}s".format(end-start))

# pack fuzzy results into DataFrame: best (top-1) score for each method
fuzzy_rows = []
for i in range(len(new)):
    row = {'new_idx': i, 'new_raw': new.iloc[i][new_text_col], 'new_clean': new.iloc[i]['text_clean']}
    for m in methods:
        top = fuzzy_results[m][i]
        if top:
            row[f'{m}_best_score'] = top[0]['score']
            row[f'{m}_best_resolved_idx'] = top[0]['resolved_idx']
            row[f'{m}_best_resolved_text'] = resolved.iloc[top[0]['resolved_idx']][resolved_text_col]
        else:
            row[f'{m}_best_score'] = 0.0
            row[f'{m}_best_resolved_idx'] = None
            row[f'{m}_best_resolved_text'] = ''
    # overall best fuzzy score across methods
    row['fuzzy_best_score'] = max(row[f'{m}_best_score'] for m in methods)
    # choose best method name
    best_m = max(methods, key=lambda mm: row[f'{mm}_best_score'])
    row['fuzzy_best_method'] = best_m
    fuzzy_rows.append(row)

fuzzy_df = pd.DataFrame(fuzzy_rows)
print("\nFuzzy results example:")
display(fuzzy_df.head(6))

# --- Bag-of-Words (CountVectorizer) and TF-IDF matching ---
print("\nComputing CountVectorizer (BoW) and TF-IDF (1-2grams) and cosine similarities...")
cv = CountVectorizer(ngram_range=(1,2), min_df=1)
tfv = TfidfVectorizer(ngram_range=(1,2), min_df=1)

# Fit on resolved corpus; transform both resolved and new
X_res_cv = cv.fit_transform(resolved['text_clean'].astype(str).tolist())
X_new_cv = cv.transform(new['text_clean'].astype(str).tolist())

X_res_tfidf = tfv.fit_transform(resolved['text_clean'].astype(str).tolist())
X_new_tfidf = tfv.transform(new['text_clean'].astype(str).tolist())

# cosine similarities (new x resolved)
cos_cv = cosine_similarity(X_new_cv, X_res_cv)
cos_tfidf = cosine_similarity(X_new_tfidf, X_res_tfidf)

# For each new, get top-3 by cosine
cv_matches = []
tfidf_matches = []
for i in range(cos_cv.shape[0]):
    top_idx_cv = np.argsort(cos_cv[i])[::-1][:3]
    cv_matches.append([{'resolved_idx': int(j), 'score': float(cos_cv[i][j]), 'resolved_text': resolved.iloc[j][resolved_text_col]} for j in top_idx_cv])
    top_idx_tf = np.argsort(cos_tfidf[i])[::-1][:3]
    tfidf_matches.append([{'resolved_idx': int(j), 'score': float(cos_tfidf[i][j]), 'resolved_text': resolved.iloc[j][resolved_text_col]} for j in top_idx_tf])

# Build tfidf/cv DataFrame summary
tf_rows = []
for i in range(len(new)):
    tf_rows.append({
        'new_idx': i,
        'new_raw': new.iloc[i][new_text_col],
        'cv_top': cv_matches[i],
        'cv_max': cv_matches[i][0]['score'],
        'tfidf_top': tfidf_matches[i],
        'tfidf_max': tfidf_matches[i][0]['score']
    })
tfidf_df = pd.DataFrame(tf_rows)
print("\nTF-IDF / BoW sample:")
display(tfidf_df.head(6))

# --- Combine fuzzy + tfidf results for export and analysis ---
combined = fuzzy_df.merge(tfidf_df[['new_idx','cv_top','cv_max','tfidf_top','tfidf_max']], on='new_idx', how='left')

# suggested resolved text: prefer fuzzy_best (token_set) then tfidf
def suggest_text(row):
    # prefer token_set (most robust)
    if row['token_set_ratio_best_score'] if 'token_set_ratio_best_score' in row else None:
        pass
    method_preference = ['token_set_ratio','token_sort_ratio','ratio','partial_ratio']
    for m in method_preference:
        key = f'{m}_best_resolved_text'
        if key in row and pd.notna(row[key]) and str(row[key]).strip() != '':
            return row[key]
    # fallback to tfidf top
    if isinstance(row['tfidf_top'], list) and len(row['tfidf_top'])>0:
        return row['tfidf_top'][0]['resolved_text']
    return ''

combined['suggested_resolved_text'] = combined.apply(suggest_text, axis=1)
# also include resolved id index (from tfidf top as fallback)
def suggested_resolved_idx(row):
    # find first available resolved_idx from fuzzy methods
    for m in methods:
        key = f'{m}_best_resolved_idx'
        if key in row and pd.notna(row[key]):
            return int(row[key])
    # fallback to tfidf top idx
    if isinstance(row['tfidf_top'], list) and len(row['tfidf_top'])>0:
        return int(row['tfidf_top'][0]['resolved_idx'])
    return None

combined['suggested_resolved_idx'] = combined.apply(suggested_resolved_idx, axis=1)

# Save combined csv
OUT_CSV = '/mnt/data/matched_results.csv'
combined.to_csv(OUT_CSV, index=False)
print("\nSaved combined matches to:", OUT_CSV)

# --- Score distributions and suggested thresholds ---
print("\nScore percentiles (fuzzy best score):")
print(np.percentile(combined['fuzzy_best_score'], [50,75,90,95]).round(2))
print("Score percentiles (tfidf_max):")
print(np.percentile(combined['tfidf_max'], [50,75,90,95]).round(4))

# Heuristic thresholds (suggestions):
print("\nHeuristic suggestions:")
print(" - fuzzy (0-100): high precision >=90, moderate >=80, recall-biased >=70")
print(" - tfidf (0-1): high precision >=0.70, moderate >=0.50, recall-biased >=0.40")

# Plot histograms for visual inspection
plt.figure(figsize=(8,3))
plt.hist(combined['fuzzy_best_score'].dropna(), bins=40)
plt.title('fuzzy_best_score distribution (0-100)')
plt.xlabel('score'); plt.ylabel('count'); plt.show()

plt.figure(figsize=(8,3))
plt.hist(combined['tfidf_max'].dropna(), bins=40)
plt.title('tfidf_max distribution (0-1)')
plt.xlabel('score'); plt.ylabel('count'); plt.show()

# --- If ground-truth label exists: evaluate and sweep thresholds ---
if new_label_col is not None:
    print("\nGround-truth label detected in 'new':", new_label_col)
    # We'll attempt to compare predicted suggested_resolved_idx with true label (assuming true label is a resolved id or resolved_idx).
    # Try to map ground truth values to resolved indices if they are ids (not indices).
    true_vals = new[new_label_col].values
    # create mapping resolved id -> resolved index
    resolved_id_to_idx = {}
    # resolved_id_col may contain numeric or string ids
    for idx, rid in zip(resolved.index, resolved[resolved_id_col].values):
        resolved_id_to_idx[str(rid)] = int(idx)
        resolved_id_to_idx[rid] = int(idx)
    # convert true to resolved_idx where possible
    true_idx = []
    for v in true_vals:
        if pd.isna(v):
            true_idx.append(None)
            continue
        # try direct index if numeric and within range
        if isinstance(v, (int, np.integer)) and 0 <= int(v) < len(resolved):
            true_idx.append(int(v)); continue
        # try mapping by id string
        key = str(v)
        if key in resolved_id_to_idx:
            true_idx.append(resolved_id_to_idx[key]); continue
        # else unknown
        true_idx.append(None)
    # attach true_idx to combined
    combined['true_resolved_idx'] = true_idx

    # Evaluation function
    def eval_thresholds_fuzzy(combined_df, method='token_set_ratio', fuzzy_thresh_list=None):
        if fuzzy_thresh_list is None:
            fuzzy_thresh_list = list(range(50, 101, 5))
        rows=[]
        score_col = f'{method}_best_score' if f'{method}_best_score' in combined_df.columns else 'fuzzy_best_score'
        for t in fuzzy_thresh_list:
            pred_idx = combined_df[score_col] >= t
            TP = ((pred_idx) & (combined_df['suggested_resolved_idx'].notna()) & (combined_df['true_resolved_idx'].notna()) & (combined_df['suggested_resolved_idx']==combined_df['true_resolved_idx'])).sum()
            FP = ((pred_idx) & (combined_df['suggested_resolved_idx'].notna()) & ((combined_df['true_resolved_idx'].isna()) | (combined_df['suggested_resolved_idx']!=combined_df['true_resolved_idx']))).sum()
            FN = ((~pred_idx) & (combined_df['true_resolved_idx'].notna())).sum()
            precision = TP/(TP+FP) if (TP+FP)>0 else np.nan
            recall = TP/(TP+FN) if (TP+FN)>0 else np.nan
            f1 = 2*precision*recall/(precision+recall) if precision and recall and (precision+recall)>0 else np.nan
            rows.append({'threshold':t,'TP':int(TP),'FP':int(FP),'FN':int(FN),'precision':precision,'recall':recall,'f1':f1})
        return pd.DataFrame(rows)

    def eval_thresholds_tfidf(combined_df, tf_thresh_list=None):
        if tf_thresh_list is None:
            tf_thresh_list = list(np.round(np.linspace(0.1,0.9,17),2))
        rows=[]
        for t in tf_thresh_list:
            pred_idx = combined_df['tfidf_max'] >= t
            TP = ((pred_idx) & (combined_df['tfidf_top'].notna()) & (combined_df['true_resolved_idx'].notna()) & 
                  (combined_df['tfidf_top'].apply(lambda x: x[0]['resolved_idx'] if isinstance(x,list) and len(x)>0 else None) == combined_df['true_resolved_idx'])).sum()
            FP = ((pred_idx) & (combined_df['tfidf_top'].notna()) & (
                  combined_df['tfidf_top'].apply(lambda x: x[0]['resolved_idx'] if isinstance(x,list) and len(x)>0 else None) != combined_df['true_resolved_idx'])).sum()
            FN = ((~pred_idx) & (combined_df['true_resolved_idx'].notna())).sum()
            precision = TP/(TP+FP) if (TP+FP)>0 else np.nan
            recall = TP/(TP+FN) if (TP+FN)>0 else np.nan
            f1 = 2*precision*recall/(precision+recall) if precision and recall and (precision+recall)>0 else np.nan
            rows.append({'threshold':t,'TP':int(TP),'FP':int(FP),'FN':int(FN),'precision':precision,'recall':recall,'f1':f1})
        return pd.DataFrame(rows)

    # run sweeps for fuzzy and tfidf
    fuzzy_eval = eval_thresholds_fuzzy(combined, method='token_set_ratio', fuzzy_thresh_list=list(range(50,101,5)))
    tfidf_eval = eval_thresholds_tfidf(combined, tf_thresh_list=list(np.round(np.linspace(0.1,0.9,17),2)))

    print("\nFuzzy threshold sweep (token_set_ratio) preview:")
    display(fuzzy_eval.head(10))
    print("\nTF-IDF threshold sweep preview:")
    display(tfidf_eval.head(10))
    # Suggest best threshold by f1
    best_fuzzy = fuzzy_eval.loc[fuzzy_eval['f1'].idxmax()]
    best_tfidf = tfidf_eval.loc[tfidf_eval['f1'].idxmax()]
    print("\nBest fuzzy threshold by F1 (token_set_ratio):", best_fuzzy.to_dict())
    print("Best TF-IDF threshold by F1:", best_tfidf.to_dict())
    # Save detailed evals
    fuzzy_eval.to_csv('/mnt/data/fuzzy_threshold_eval.csv', index=False)
    tfidf_eval.to_csv('/mnt/data/tfidf_threshold_eval.csv', index=False)
    print("\nSaved fuzzy and tfidf evaluation CSVs to /mnt/data/")

else:
    print("\nNo ground-truth label column found in 'new'. If you have labels, name the column like 'resolved_id' or 'match_id' and re-run to get automatic evaluation.")

print("\nDone. Check /mnt/data/matched_results.csv for full output.")


rapidfuzz available: True
resolved.shape: (5, 2)
new.shape: (20, 2)
Using resolved text column: Pre_Resolved_Query
Using new text column: Variation_Query
Resolved id column: Query_ID
New ground-truth label column (if any): Matches_With_Query_ID

Sample cleaned resolved:
 0              unable connect internet
1       payment failed during checkout
2    app crashes when opening settings
3         forgot password unable reset
4           unable upload files server
Name: text_clean, dtype: object

Sample cleaned new:
 0               unabel conect internet
1                 can connect internet
2                      intenet working
3         payment failed while chekout
4    payment go through during chckout
Name: text_clean, dtype: object
Using fuzzy function: rapidfuzz

Computing fuzzy top-3 for each method... (this may take a while for large datasets)


  0%|          | 0/20 [00:00<?, ?it/s]

Fuzzy matching completed in 0.0s

Fuzzy results example:


,new_idx,new_raw,new_clean,token_set_ratio_best_score,token_set_ratio_best_resolved_idx,token_set_ratio_best_resolved_text,token_sort_ratio_best_score,token_sort_ratio_best_resolved_idx,token_sort_ratio_best_resolved_text,ratio_best_score,ratio_best_resolved_idx,ratio_best_resolved_text,partial_ratio_best_score,partial_ratio_best_resolved_idx,partial_ratio_best_resolved_text,fuzzy_best_score,fuzzy_best_method
0,0,Unabel to conect to the internet,unabel conect internet,93.333333,0,Unable to connect to the internet,93.333333,0,Unable to connect to the internet,93.333333,0,Unable to connect to the internet,90.909091,0,Unable to connect to the internet,93.333333,token_set_ratio
1,1,Can’t connect to internet,can connect internet,88.888889,0,Unable to connect to the internet,74.418605,0,Unable to connect to the internet,83.720930,0,Unable to connect to the internet,91.891892,0,Unable to connect to the internet,91.891892,partial_ratio
2,2,Intenet not working,intenet working,47.368421,0,Unable to connect to the internet,47.368421,0,Unable to connect to the internet,42.105263,0,Unable to connect to the internet,60.869565,0,Unable to connect to the internet,60.869565,partial_ratio
3,3,Payment failed while chekout,payment failed while chekout,82.758621,1,Payment failed during checkout,75.862069,1,Payment failed during checkout,82.758621,1,Payment failed during checkout,78.571429,1,Payment failed during checkout,82.758621,token_set_ratio
4,4,Payment did not go through during chckout,payment go through during chckout,73.015873,1,Payment failed during checkout,73.015873,1,Payment failed during checkout,73.015873,1,Payment failed during checkout,66.666667,1,Payment failed during checkout,73.015873,token_sort_ratio
5,5,Payment issue at check out,payment issue check out,60.377358,1,Payment failed during checkout,60.377358,1,Payment failed during checkout,71.698113,1,Payment failed during checkout,60.000000,1,Payment failed during checkout,71.698113,ratio



Computing CountVectorizer (BoW) and TF-IDF (1-2grams) and cosine similarities...

TF-IDF / BoW sample:


,new_idx,new_raw,cv_top,cv_max,tfidf_top,tfidf_max
0,0,Unabel to conect to the internet,"[{'resolved_idx': 0, 'score': 0.44721359549995...",0.447214,"[{'resolved_idx': 0, 'score': 0.47412464855584...",0.474125
1,1,Can’t connect to internet,"[{'resolved_idx': 0, 'score': 0.77459666924148...",0.774597,"[{'resolved_idx': 0, 'score': 0.82120798041946...",0.821208
2,2,Intenet not working,"[{'resolved_idx': 4, 'score': 0.0, 'resolved_t...",0.000000,"[{'resolved_idx': 4, 'score': 0.0, 'resolved_t...",0.000000
3,3,Payment failed while chekout,"[{'resolved_idx': 1, 'score': 0.65465367070797...",0.654654,"[{'resolved_idx': 1, 'score': 0.65465367070797...",0.654654
4,4,Payment did not go through during chckout,"[{'resolved_idx': 1, 'score': 0.53452248382484...",0.534522,"[{'resolved_idx': 1, 'score': 0.53452248382484...",0.534522
5,5,Payment issue at check out,"[{'resolved_idx': 1, 'score': 0.37796447300922...",0.377964,"[{'resolved_idx': 1, 'score': 0.37796447300922...",0.377964


OSError: Cannot save file into a non-existent directory: '/mnt/data'

In [6]:
# Install required libraries (already installed, but included for completeness)
!pip install fuzzywuzzy python-levenshtein scikit-learn nltk

import pandas as pd
import string
import nltk
from nltk.stem import PorterStemmer
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt')

# Load the CSV files (REPLACE with your actual paths, e.g., '/kaggle/input/query-dataset/resolved_queries.csv')
resolved_path = '/kaggle/input/text-assignment/resolved_queries.csv'  # e.g., replace 'your-dataset'
unresolved_path = '/kaggle/input/text-assignment/new_queries.csv' # Fix this
df_resolved = pd.read_csv(resolved_path)
df_unresolved = pd.read_csv(unresolved_path)

# Debug prints (for reference)
print("Resolved CSV Columns:", df_resolved.columns.tolist())
print("Resolved CSV Head:\n", df_resolved.head())
print("\nUnresolved CSV Columns:", df_unresolved.columns.tolist())
print("Unresolved CSV Head:\n", df_unresolved.head())

# Extract queries using correct columns
resolved_queries = df_resolved['Pre_Resolved_Query'].fillna('').tolist()  # Handle NaN
unresolved_queries = df_unresolved['Variation_Query'].fillna('').tolist()
ground_truth_ids = df_unresolved['Matches_With_Query_ID'].tolist()  # For evaluation

# Preprocessing function
stemmer = PorterStemmer()
def preprocess(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    stemmed = [stemmer.stem(word) for word in tokens]
    return ' '.join(stemmed)

# Apply preprocessing
resolved_preprocessed = [preprocess(q) for q in resolved_queries]
unresolved_preprocessed = [preprocess(q) for q in unresolved_queries]

# Fuzzy match function: Returns list of (unresolved, matched_resolved, score, matched_id)
def fuzzy_match(unresolved, resolved_list, df_resolved, method='token_set_ratio', threshold=85):
    matches = []
    for idx, query in enumerate(unresolved):
        if not query:
            matches.append((query, None, 0, None))
            continue
        if method == 'ratio':
            best_match, score = process.extractOne(query, resolved_list, scorer=fuzz.ratio)
        elif method == 'partial_ratio':
            best_match, score = process.extractOne(query, resolved_list, scorer=fuzz.partial_ratio)
        elif method == 'token_sort_ratio':
            best_match, score = process.extractOne(query, resolved_list, scorer=fuzz.token_sort_ratio)
        elif method == 'token_set_ratio':
            best_match, score = process.extractOne(query, resolved_list, scorer=fuzz.token_set_ratio)
        else:
            raise ValueError("Invalid fuzzy method")
        
        # Manually find the index of the best match
        best_idx = resolved_list.index(best_match) if best_match in resolved_list else -1
        matched_id = df_resolved.iloc[best_idx]['Query_ID'] if score >= threshold and best_idx != -1 else None
        matches.append((query, best_match if score >= threshold else None, score, matched_id))
    return matches

# Evaluate accuracy: Compare matched_id to ground_truth_ids
def evaluate_accuracy(matches, ground_truth):
    correct = sum(1 for (u, r, s, m_id), gt_id in zip(matches, ground_truth) if m_id == gt_id)
    return correct / len(ground_truth) if len(ground_truth) > 0 else 0

# Test multiple thresholds for fuzzy
fuzzy_thresholds = [70, 80, 85, 90]
methods = ['ratio', 'partial_ratio', 'token_sort_ratio', 'token_set_ratio']

for method in methods:
    print(f"\nEvaluating Fuzzy Method: {method}")
    for thresh in fuzzy_thresholds:
        matches = fuzzy_match(unresolved_preprocessed, resolved_preprocessed, df_resolved, method=method, threshold=thresh)
        accuracy = evaluate_accuracy(matches, ground_truth_ids)
        print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}")
        print("Sample Matches:")
        for unresolved, resolved, score, m_id in matches[:5]:
            print(f"Unresolved: {unresolved} -> Resolved: {resolved} (Score: {score}, Matched ID: {m_id})")

# Vector match function: Returns list of (unresolved, matched_resolved, score, matched_id)
def vector_match(unresolved, resolved, df_resolved, vectorizer_type='tfidf', threshold=0.75):
    all_queries = [q for q in resolved + unresolved if q]
    if not all_queries:
        return []
    if vectorizer_type == 'bow':
        vectorizer = CountVectorizer(stop_words='english')
    elif vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(stop_words='english')
    else:
        raise ValueError("Invalid vectorizer type")
    
    vectors = vectorizer.fit_transform(all_queries)
    resolved_vectors = vectors[:len(resolved)]
    unresolved_vectors = vectors[len(resolved):]
    
    similarities = cosine_similarity(unresolved_vectors, resolved_vectors)
    matches = []
    for i, sim_row in enumerate(similarities):
        max_sim_idx = sim_row.argmax()
        max_sim = sim_row[max_sim_idx]
        matched_id = df_resolved.iloc[max_sim_idx]['Query_ID'] if max_sim >= threshold else None
        matched_res = resolved[max_sim_idx] if max_sim >= threshold else None
        matches.append((unresolved[i], matched_res, max_sim, matched_id))
    return matches

# Test multiple thresholds for vectors
vector_thresholds = [0.6, 0.7, 0.75, 0.8]

print("\nEvaluating BoW")
for thresh in vector_thresholds:
    matches = vector_match(unresolved_preprocessed, resolved_preprocessed, df_resolved, vectorizer_type='bow', threshold=thresh)
    accuracy = evaluate_accuracy(matches, ground_truth_ids)
    print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}")
    print("Sample Matches:")
    for unresolved, resolved, score, m_id in matches[:5]:
        print(f"Unresolved: {unresolved} -> Resolved: {resolved} (Score: {score}, Matched ID: {m_id})")

print("\nEvaluating TF-IDF")
for thresh in vector_thresholds:
    matches = vector_match(unresolved_preprocessed, resolved_preprocessed, df_resolved, vectorizer_type='tfidf', threshold=thresh)
    accuracy = evaluate_accuracy(matches, ground_truth_ids)
    print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}")
    print("Sample Matches:")
    for unresolved, resolved, score, m_id in matches[:5]:
        print(f"Unresolved: {unresolved} -> Resolved: {resolved} (Score: {score}, Matched ID: {m_id})")

# Save best matches (e.g., TF-IDF at threshold 0.75) to CSV
best_matches = vector_match(unresolved_preprocessed, resolved_preprocessed, df_resolved, vectorizer_type='tfidf', threshold=0.75)
df_matches = pd.DataFrame(best_matches, columns=['unresolved_preprocessed', 'matched_resolved', 'score', 'matched_id'])
df_matches['original_unresolved'] = unresolved_queries
df_matches['ground_truth_id'] = ground_truth_ids
df_matches.to_csv('/kaggle/working/matches.csv', index=False)
print("\nMatches saved to /kaggle/working/matches.csv")

Resolved CSV Columns: ['Query_ID', 'Pre_Resolved_Query']
Resolved CSV Head:
    Query_ID                    Pre_Resolved_Query
0         1     Unable to connect to the internet
1         2        Payment failed during checkout
2         3     App crashes when opening settings
3         4   Forgot password and unable to reset
4         5  Unable to upload files to the server

Unresolved CSV Columns: ['Variation_Query', 'Matches_With_Query_ID']
Unresolved CSV Head:
                              Variation_Query  Matches_With_Query_ID
0           Unabel to conect to the internet                      1
1                  Can’t connect to internet                      1
2                        Intenet not working                      1
3               Payment failed while chekout                      2
4  Payment did not go through during chckout                      2

Evaluating Fuzzy Method: ratio
Threshold 70: Accuracy = 0.50
Sample Matches:
Unresolved: unabel to conect to the internet 

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
matches=pd.read_csv('/kaggle/working/matches.csv')
matches

,unresolved_preprocessed,matched_resolved,score,matched_id,original_unresolved,ground_truth_id
0,unabel to conect to the internet,NaN,0.266627,NaN,Unabel to conect to the internet,1
1,can ’ t connect to internet,unabl to connect to the internet,0.845594,1.0,Can’t connect to internet,1
2,intenet not work,NaN,0.000000,NaN,Intenet not working,1
3,payment fail while chekout,NaN,0.448407,NaN,Payment failed while chekout,2
4,payment did not go through dure chckout,NaN,0.433791,NaN,Payment did not go through during chckout,2
5,payment issu at check out,NaN,0.188484,NaN,Payment issue at check out,2
6,applic crash when open sete,NaN,0.468542,NaN,Application crashes when opening setings,3
7,app crash when go to set,app crash when open set,0.823581,3.0,App crash when going to settings,3
8,set caus the app to chrash,NaN,0.401039,NaN,Settings cause the app to chrash,3
9,forgot passwrd and cant reset,NaN,0.554582,NaN,Forgot passwrd and cant reset,4
